In [ ]:
#@title Gene Expression Bootstrapper { display-mode: "form" }
from IPython.display import display, HTML
display(HTML("""
<div style='font-family:sans-serif;'>

  <div style='padding:0.7rem 1rem;background:#f8f9fb;border:1px solid #e0e0e0;border-radius:6px;margin-bottom:1.5rem;font-size:13px;color:#666;line-height:1.7;'>
    Copyright &copy; 2026. All rights reserved.<br>
    This program is free of charge for academic research and education purposes.
    For commercial use or licensing inquiries, please contact the authors
    (<a href='mailto:yeohc@a-star.edu.sg' style='color:#1a56db;'>yeohc@a-star.edu.sg</a> or
    <a href='mailto:yeodynasty@yahoo.com.sg' style='color:#1a56db;'>yeodynasty@yahoo.com.sg</a>).
    Refer to the LICENSE file in the root directory for full terms and conditions.
  </div>

  <div style='display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:1.5rem;'>
    <div style='flex:1;'>
      <h1 style='margin:0 0 0.5rem 0;font-size:2rem;font-weight:700;'>Gene Expression Bootstrapper</h1>
      <p style='margin:0;color:#444;font-size:16px;'>Bootstraps missing gene expression values for unmapped and unknown genes in metatranscriptomics data, sampling from genes in the same metabolic subsystem. Output files serve as the geneExpr folder input for FEA.</p>
    </div>
    <div style='margin-left:2rem;flex-shrink:0;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/proteinSeqMapping/main/bii_horizontal_logo_smalle472014210204e8886d5cf09c653e7e5.png' style='width:380px;'/>
    </div>
  </div>

  <hr style='border:none;border-top:1px solid #ddd;margin:1.5rem 0;'/>

  <h2 style='font-size:1.4rem;margin:0 0 0.8rem 0;'>Instructions:</h2>
  <ol style='margin:0 0 1.5rem 0;padding-left:1.5rem;font-size:16px;line-height:1.8;'>
    <li><strong>Optional:</strong> adjust the batch count in Step 1 (<strong>default: 1000 recommended</strong>).</li>
    <li>Go to <strong>Runtime &#8594; Run all</strong> in the top menu.</li>
    <li>Upload your files when prompted in Step 2. Sample files are available to download next to each upload prompt.</li>
    <li>The output ZIP will download automatically when complete.</li>
  </ol>

</div>
"""))

In [ ]:
#@title Step 1 — Set up environment { display-mode: "form" }
import subprocess, sys
from IPython.display import display, HTML

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#1a56db;'>&#9881; Setting up environment...</b>
  <p style='margin:0.3rem 0 0 0;font-size:16px;color:#555;'>This runs automatically and takes about 1 minute.</p>
</div>
"""))

import os
if not os.path.isdir('/content/Gene-Expression-Bootstrapper'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/MeMoModelling/Gene-Expression-Bootstrapper.git',
                    '/content/Gene-Expression-Bootstrapper'], check=True, capture_output=True)
sys.path.insert(0, '/content/Gene-Expression-Bootstrapper')

subprocess.run(['pip', 'install', '-q', 'pandas', 'numpy', 'openpyxl', 'python-libsbml'], capture_output=True)

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0fff8;border-left:4px solid #006d5b;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#006d5b;'>&#10003; Ready! Proceed to the next step.</b>
</div>
"""))

In [ ]:
#@title Step 1 — Adjust batch count { display-mode: "form" }
from IPython.display import display, HTML
from google.colab import output as colab_output

display(HTML("""
<div style='font-family:sans-serif;padding:1rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;'>
  <p style='margin:0 0 0.8rem 0;font-size:16px;'><b>Optional:</b> Number of bootstrap iterations (= number of output CSV files). <b>Default: 1000 recommended.</b></p>
  <div style='display:flex;align-items:center;gap:1rem;'>
    <input type='range' id='batchSlider' min='100' max='5000' step='100' value='1000'
      style='flex:1;height:4px;accent-color:#1a56db;cursor:pointer;border-radius:4px;'>
    <span id='batchVal' style='font-size:1.1rem;font-weight:700;color:#1a56db;min-width:48px;text-align:right;'>1000</span>
  </div>
</div>
<script>
  document.getElementById('batchSlider').addEventListener('input', function() {
    document.getElementById('batchVal').textContent = this.value;
  });
</script>
"""))

batch_count = int(colab_output.eval_js("document.getElementById('batchSlider').value"))

In [ ]:
#@title Step 2 — Upload files & run { display-mode: "form" }
from IPython.display import display, HTML
from google.colab import files
import os, sys, zipfile, base64, io
import pandas as pd

sys.path.insert(0, '/content/Gene-Expression-Bootstrapper')

# ── Sample file helpers ──
def _xlsx_b64(sheets):
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine='openpyxl') as w:
        for name, df in sheets.items():
            df.to_excel(w, sheet_name=name, index=False)
    return base64.b64encode(buf.getvalue()).decode()

def _csv_b64(df):
    return base64.b64encode(df.to_csv(index=False).encode()).decode()

def _btn(b64, filename, mime):
    return (f"<a href='data:{mime};base64,{b64}' download='{filename}' "
            f"style='display:inline-block;padding:0.35rem 0.85rem;background:#f0f4ff;"
            f"color:#1a56db;border:1.5px solid #1a56db;border-radius:4px;text-decoration:none;"
            f"font-family:sans-serif;font-size:14px;margin-top:0.4rem;'>&#8659; Download {filename}</a>")

_model_b64 = _xlsx_b64({'Reactions': pd.DataFrame({
    'id':     ['R_PFK',                'R_CS'],
    'name':   ['Phosphofructokinase',  'Citrate synthase'],
    'system': ['Glycolysis',           'TCA Cycle'],
    'genes':  ['gene_001 or gene_002', 'gene_003'],
})})

_mapping_b64 = _xlsx_b64({'Model': pd.DataFrame({
    'model_tag':   ['gene_001', 'gene_002', 'gene_003'],
    'has_mapping': [True,        True,       False],
    'gene_id':     ['NCBI_001', 'NCBI_002', ''],
})})

_geneexpr_b64 = _csv_b64(pd.DataFrame({
    'gene_id':  ['NCBI_001', 'NCBI_002', 'NCBI_003'],
    'sample_1': [12.5,        0.0,        7.3],
    'sample_2': [8.1,         3.2,        0.0],
}))

# ── Upload model files ──
display(HTML(f"""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Upload your prefixed model files</b>
  <p style='margin:0.3rem 0 0.4rem 0;font-size:15px;color:#555;'>One <code>.xlsx</code> per species &mdash; e.g. <code>species_model.xlsx</code></p>
  {_btn(_model_b64, 'sample_model.xlsx', 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')}
</div>
"""))
uploaded_models = files.upload()
model_filenames = list(uploaded_models.keys())
display(HTML(f"<p style='font-family:sans-serif;font-size:15px;color:#006d5b;'>&#10003; Uploaded: {', '.join('<code>' + f + '</code>' for f in model_filenames)}</p>"))

# ── Upload mapping files ──
display(HTML(f"""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Upload your identifier mapping files</b>
  <p style='margin:0.3rem 0 0.4rem 0;font-size:15px;color:#555;'>One <code>.xlsx</code> per species, <strong>in the same order</strong> as the model files &mdash; e.g. <code>species_mapping.xlsx</code></p>
  {_btn(_mapping_b64, 'sample_mapping.xlsx', 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')}
</div>
"""))
uploaded_mappings = files.upload()
mapping_filenames = list(uploaded_mappings.keys())
display(HTML(f"<p style='font-family:sans-serif;font-size:15px;color:#006d5b;'>&#10003; Uploaded: {', '.join('<code>' + f + '</code>' for f in mapping_filenames)}</p>"))

# ── Upload gene expression CSV ──
display(HTML(f"""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Upload your combined gene expression CSV</b>
  <p style='margin:0.3rem 0 0.4rem 0;font-size:15px;color:#555;'><code>combined_geneExpr.csv</code></p>
  {_btn(_geneexpr_b64, 'sample_combined_geneExpr.csv', 'text/csv')}
</div>
"""))
uploaded_geneexpr = files.upload()
geneexpr_filename = list(uploaded_geneexpr.keys())[0]
display(HTML(f"<p style='font-family:sans-serif;font-size:15px;color:#006d5b;'>&#10003; Uploaded: <code>{geneexpr_filename}</code></p>"))

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#1a56db;font-size:16px;'>&#8635; Processing&hellip; this may take several minutes depending on batch count.</b>
</div>
"""))

species_prefixes = [f.split('_')[0] for f in model_filenames]

from utils.bootstrap_genes import bootstrap_genes

output_dir = 'geneExpr_output'
os.makedirs(output_dir, exist_ok=True)

bootstrap_genes(
    model_pre_filenames=model_filenames,
    mapping_filenames=mapping_filenames,
    species_prefixes=species_prefixes,
    combined_geneExpr_filename=geneexpr_filename,
    geneExpr_folder=output_dir,
    batch_count=batch_count,
)

zip_name = 'geneExpr_bootstrapped.zip'
output_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.csv')])
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in output_files:
        zf.write(os.path.join(output_dir, fname), arcname=fname)

display(HTML(f"""
<div style='padding:1rem 1.2rem;background:#f0fff8;border:1.5px solid #006d5b;border-radius:8px;font-family:sans-serif;margin-top:0.5rem;'>
  <b style='font-size:16px;color:#006d5b;'>&#10003; Done!</b>
  <p style='margin:0.4rem 0 0 0;font-size:16px;color:#333;'>{len(output_files)} bootstrapped files generated.</p>
  <p style='margin:0.4rem 0 0 0;font-size:16px;color:#333;'>Downloading <code>{zip_name}</code>&hellip;</p>
  <p style='margin:0.6rem 0 0 0;font-size:16px;color:#555;'>&#128193; If the download doesn't start automatically, find <code>{zip_name}</code> in the <b>Files panel on the left</b> and download manually.</p>
</div>
"""))

files.download(zip_name)